## PPK with knn and svm

In [ ]:
# !pip install -q scikit-learn matplotlib seaborn pandas numpy
import os
import re
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from numpy import pi as PI, exp, power
from numpy.linalg import det, inv
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    precision_score, recall_score, f1_score,
)
warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Numba JIT — accelerates the innermost Gaussian similarity evaluation.
# Falls back gracefully if Numba is not installed.
# ---------------------------------------------------------------------------
try:
    from numba import njit

    @njit(cache=True)
    def _ppk_gauss_sim_numba(mu1, S1_inv, mu2, S2_inv, S1_det, S2_det, rho):
        D = mu1.shape[0]
        S_combined_inv = rho * S1_inv + rho * S2_inv
        S_combined     = np.linalg.inv(S_combined_inv)
        mu_combined    = rho * (S1_inv @ mu1 + S2_inv @ mu2)

        det_combined = np.linalg.det(S_combined)
        if det_combined <= 0 or S1_det <= 0 or S2_det <= 0:
            return 1e-10

        z = (2 * np.pi) ** ((1 - 2 * rho) * D / 2) \
            * det_combined ** 0.5 \
            * S1_det ** (-rho / 2) \
            * S2_det ** (-rho / 2)

        q = (-rho / 2 * (mu1 @ S1_inv @ mu1)
             - rho / 2 * (mu2 @ S2_inv @ mu2)
             + 0.5 * (mu_combined @ S_combined @ mu_combined))

        if not np.isfinite(q) or not np.isfinite(z):
            return 1e-10
        return z * np.exp(q)

    _NUMBA = True
    print("Numba available — PPK kernel evaluations will use JIT compilation.")
except ImportError:
    _NUMBA = False
    print("Numba not found — falling back to NumPy (slower). "
          "Install with: pip install numba")


# =============================================================================
# Configuration
# =============================================================================

# ---- Paths -------------------------------------------------------------
CSV_PATH    = ".../hmm_results/N5G5/hmm_results.csv"
OUTPUT_BASE = ".../hmm_results/ppk_classifiers"

# ---- HMM model parameters (must match CSV) -----------------------------
N_STATES     = 5
N_COMPONENTS = 5
N_FEATURES   = None   # None = auto-detected from CSV

# ---- Config tag (auto-derived; same pattern as RKHS script) ------------
_match     = re.search(r'(N\d+G\d+)', CSV_PATH)
CONFIG_TAG = _match.group(1) if _match else f"N{N_STATES}G{N_COMPONENTS}"


# =============================================================================
# Checkpoint manager (identical to rkhs_classifiers.py)
#
# Saves progress after every completed outer fold and every PERM_CKPT_EVERY
# permutations, so a Kaggle session timeout does not lose completed work.
# On restart, automatically resumes from the next un-run fold/permutation.
# =============================================================================

PERM_CKPT_EVERY = 100   # save permutation test progress every N permutations


class CheckpointManager:
    def __init__(self, output_dir, classifier):
        self.path = os.path.join(output_dir, f"checkpoint_{classifier}.pkl")

    def load(self):
        """Returns the saved state dict, or None if no checkpoint exists."""
        if os.path.exists(self.path):
            try:
                with open(self.path, 'rb') as f:
                    state = pickle.load(f)
                print(f"  [checkpoint] Found existing checkpoint: {self.path}")
                return state
            except Exception as exc:
                print(f"  [checkpoint] Failed to load checkpoint, "
                      f"starting fresh: {exc}")
                return None
        return None

    def save(self, state):
        """Atomically save state to disk (write to temp file, then rename)."""
        tmp_path = self.path + ".tmp"
        with open(tmp_path, 'wb') as f:
            pickle.dump(state, f)
        os.replace(tmp_path, self.path)

    def clear(self):
        if os.path.exists(self.path):
            os.remove(self.path)

# ---- Nested CV structure (identical to RKHS script) ---------------------
N_OUTER_SPLITS  = 5
N_OUTER_REPEATS = 5
N_INNER_SPLITS  = 3
OUTER_CV_BASE_SEED = 0

# ---- Hyperparameter grid (Option A — 7 points per kernel-defining param) -
# T   : number of HMM transition propagation steps in the PPK recursion
# rho : PPK kernel exponent (state-mixing weight), rho in (0, 1]
# C   : SVM-only regularisation parameter. PPK-SVM uses K_norm directly
#       as the kernel (it is already a bounded [0,1] similarity / Gram
#       matrix), so no separate gamma envelope parameter is needed here
#       — unlike RKHS, where MMD2 is an unbounded distance requiring an
#       exp(-D/2*gamma^2) conversion to obtain a kernel.
# k   : KNN-only, number of neighbours
T_GRID     = [1, 2, 3, 5, 7, 10, 15]
RHO_GRID   = [0.1, 0.25, 0.4, 0.55, 0.7, 0.85, 1.0]
C_GRID     = [0.01, 0.1, 1.0, 10.0, 100.0]
K_GRID     = [1, 3, 5, 7, 9]

KERNEL_EPS = 1e-10   # SVM PSD regularisation, same as RKHS script

# ---- Permutation test ----------------------------------------------------
N_PERMUTATIONS   = 1000
PERMUTATION_SEED = 0

# ---- Bootstrap CI ----------------------------------------------------------
N_BOOTSTRAP = 2000
CI_ALPHA    = 0.95

# ---- Column names ----------------------------------------------------------
COL_SUBJECT_TYPE = "subject_type"
COL_SUBJECT_ID   = "subject_id"
ADHD_LABEL       = "ADHD"

# ---- PPK regularisation -----------------------------------------------------
PPK_COV_REG = 1e-6


# =============================================================================
# PPK kernel functions
#
# Notation:
#   rho (ρ) — PPK exponent, controls the mixing weight between the two
#             distributions in the probability product kernel.
#   T        — number of HMM transition propagation steps.
#
# This kernel operates directly between HMM states using the INITIAL
# state distribution pi_0 (not the stationary distribution), and uses
# only the single best (highest-weight) GMM component per state —
# matching the original PPK formulation as a kernel between HMMs.
# =============================================================================

def ppk_gaussian_similarity_numpy(mu1, Sigma1, mu2, Sigma2, rho, reg=PPK_COV_REG):
    """NumPy fallback for the PPK Gaussian similarity between two states."""
    D = mu1.shape[0]
    S1 = Sigma1 + reg * np.eye(D)
    S2 = Sigma2 + reg * np.eye(D)
    try:
        S1_inv = inv(S1)
        S2_inv = inv(S2)
        S_combined = inv(rho * S1_inv + rho * S2_inv)
    except np.linalg.LinAlgError:
        return 1e-10

    mu_combined = rho * (S1_inv @ mu1 + S2_inv @ mu2)

    try:
        det_combined = det(S_combined)
        det1, det2   = det(S1), det(S2)
        if det_combined <= 0 or det1 <= 0 or det2 <= 0:
            return 1e-10
        z = power(2 * PI, (1 - 2 * rho) * D / 2) \
            * power(det_combined, 0.5) \
            * power(det1, -rho / 2) \
            * power(det2, -rho / 2)
    except Exception:
        return 1e-10

    q = (-rho / 2 * (mu1 @ S1_inv @ mu1)
         - rho / 2 * (mu2 @ S2_inv @ mu2)
         + 0.5 * (mu_combined @ S_combined @ mu_combined))

    if not np.isfinite(q) or not np.isfinite(z):
        return 1e-10
    return float(z * exp(q))


def ppk_gaussian_similarity(mu1, Sigma1, mu2, Sigma2, rho, reg=PPK_COV_REG):
    """PPK Gaussian similarity with Numba JIT when available."""
    if not _NUMBA:
        return ppk_gaussian_similarity_numpy(mu1, Sigma1, mu2, Sigma2, rho, reg)
    D = mu1.shape[0]
    S1 = Sigma1 + reg * np.eye(D)
    S2 = Sigma2 + reg * np.eye(D)
    try:
        S1_inv, S1_det = inv(S1), det(S1)
        S2_inv, S2_det = inv(S2), det(S2)
    except np.linalg.LinAlgError:
        return 1e-10
    return float(_ppk_gauss_sim_numba(mu1, S1_inv, mu2, S2_inv,
                                      S1_det, S2_det, rho))


def ppk_kernel(hmm1, hmm2, T, rho):
    """
    Probability Product Kernel between two HMMs.

    Uses the initial state distribution pi (not stationary), and
    propagates the state-similarity matrix Psi through T transition
    steps via the respective transition matrices A1, A2.

    k(HMM1, HMM2; T, rho) = sum( alpha_T )
    """
    pi1, A1, mu1, Sigma1 = hmm1['pi'], hmm1['A'], hmm1['mu'], hmm1['Sigma']
    pi2, A2, mu2, Sigma2 = hmm2['pi'], hmm2['A'], hmm2['mu'], hmm2['Sigma']
    M, N = pi1.shape[0], pi2.shape[0]

    Psi = np.zeros((M, N))
    for i in range(M):
        for j in range(N):
            Psi[i, j] = ppk_gaussian_similarity(
                mu1[i], Sigma1[i], mu2[j], Sigma2[j], rho
            )

    alpha = np.power(np.outer(pi1, pi2), rho) * Psi
    for _ in range(T):
        alpha = A1.T @ alpha @ A2
        alpha = np.power(alpha, rho) * Psi

    return float(np.sum(alpha))


def compute_similarity_matrix(hmms, T, rho, verbose=True):
    """
    Compute the normalised n×n PPK similarity matrix (values in [0,1]).

    Normalisation: K_norm[i,j] = K[i,j] / sqrt(K[i,i] * K[j,j])
    """
    n = len(hmms)
    K = np.zeros((n, n))
    if verbose:
        print(f"    PPK similarity matrix: n={n}  T={T}  rho={rho:.4g} ...")

    for i in range(n):
        for j in range(i, n):
            K[i, j] = ppk_kernel(hmms[i], hmms[j], T=T, rho=rho)
            K[j, i] = K[i, j]

    K_norm = np.zeros_like(K)
    for i in range(n):
        for j in range(n):
            denom = np.sqrt(max(K[i, i] * K[j, j], 0.0))
            K_norm[i, j] = K[i, j] / denom if denom > 1e-10 else 0.0
    K_norm = np.clip(K_norm, 0, 1)

    if verbose:
        print(f"    Done. K_norm min={K_norm.min():.4g}  "
              f"max={K_norm.max():.4g}  mean={K_norm.mean():.4g}")
    return K_norm


def similarity_to_distance(K_norm):
    """Convert normalised PPK similarity to a distance matrix for KNN."""
    D = np.maximum(1.0 - K_norm, 0.0)
    np.fill_diagonal(D, 0.0)
    return D


def kernel_to_svm_kernel(K_norm, kernel_eps=KERNEL_EPS):
    """
    Regularise the PPK similarity matrix for use directly as an SVM kernel.

    Unlike RKHS (where MMD2 is an unbounded DISTANCE and must be converted
    to a kernel via exp(-D/2*gamma^2)), the normalised PPK matrix K_norm
    is ALREADY a bounded [0,1] similarity / Gram matrix. Re-exponentiating
    it through an RBF-style transform is unnecessary, introduces a spurious
    gamma hyperparameter not part of the standard PPK-SVM formulation, and
    at low gamma collapses the kernel toward a near-constant matrix
    (causing the SVM to degenerate to majority-class prediction).

    K_norm is used directly as the SVM kernel here, with only the same
    PSD-correction approach as the RKHS script:
        1. Eigenvalue flooring — clip negative eigenvalues to 0
        2. Diagonal shift       — add kernel_eps * I
    """
    K = K_norm.copy()
    eigvals, eigvecs = np.linalg.eigh(K)
    if (eigvals < 0).any():
        eigvals = np.maximum(eigvals, 0.0)
        K       = eigvecs @ np.diag(eigvals) @ eigvecs.T
        K       = (K + K.T) / 2.0
    K += kernel_eps * np.eye(K.shape[0])
    return K


def check_kernel_psd(K):
    eigvals    = np.linalg.eigvalsh(K)
    min_eig    = float(eigvals.min())
    max_eig    = float(eigvals.max())
    n_negative = int((eigvals < 0).sum())
    condition  = max_eig / max(abs(min_eig), 1e-12)
    print(f"    Kernel PSD check: min_eig={min_eig:.3e}  max_eig={max_eig:.3e}  "
          f"n_negative={n_negative}  condition={condition:.3e}")
    return min_eig >= -1e-10


# =============================================================================
# CSV loading — PPK-specific parameter extraction
#
# Differs from the RKHS script:
#   - Reads INITIAL pi (pi_{i} columns), not stationary_pi.
#   - Collapses each state's GMM to its single best (highest-weight)
#     component, matching the original PPK formulation.
# =============================================================================

def infer_n_features(df):
    idxs = []
    for c in df.columns:
        if c.startswith("gmm_mean_0_0_f"):
            try:
                idxs.append(int(c.split('f')[-1]))
            except ValueError:
                pass
    return (max(idxs) + 1) if idxs else 1


def load_hmm_from_csv_row(row, n_states, n_components, n_features):
    """
    Load one HMM from a CSV row for PPK use:
        pi    : initial state distribution (pi_{i} columns)
        A     : transition matrix
        mu    : best-component mean per state
        Sigma : best-component covariance per state
    """
    pi = np.array([row[f'pi_{i}'] for i in range(n_states)], dtype=float)
    pi = np.maximum(pi, 0.0)
    s  = pi.sum()
    pi = pi / s if s > 1e-12 else np.ones(n_states) / n_states

    A = np.zeros((n_states, n_states), dtype=float)
    for i in range(n_states):
        for j in range(n_states):
            A[i, j] = row[f'A_{i}{j}']
    rs = A.sum(axis=1, keepdims=True)
    rs[rs == 0] = 1.0
    A = A / rs

    mu    = np.zeros((n_states, n_features), dtype=float)
    Sigma = np.zeros((n_states, n_features, n_features), dtype=float)

    for state in range(n_states):
        weights   = [row[f'gmm_weight_{state}_{k}'] for k in range(n_components)]
        best_comp = int(np.argmax(weights))

        for feat in range(n_features):
            mu[state, feat] = row[f'gmm_mean_{state}_{best_comp}_f{feat}']

        for i in range(n_features):
            for j in range(n_features):
                if i <= j:
                    Sigma[state, i, j] = row[f'gmm_cov_{state}_{best_comp}_f{i}f{j}']
                else:
                    Sigma[state, i, j] = Sigma[state, j, i]

    return {'pi': pi, 'A': A, 'mu': mu, 'Sigma': Sigma}


def load_dataset(csv_path, n_states, n_components, n_features=None):
    print(f"Loading: {csv_path}")
    df = pd.read_csv(csv_path)
    if n_features is None:
        n_features = infer_n_features(df)
        print(f"  Auto-detected n_features = {n_features}")

    n_adhd    = (df[COL_SUBJECT_TYPE].str.upper() == ADHD_LABEL.upper()).sum()
    n_control = len(df) - n_adhd
    print(f"  Total={len(df)}  ADHD={n_adhd}  CONTROL={n_control}")

    hmms, labels, sids = [], [], []
    for idx, row in df.iterrows():
        try:
            hmm = load_hmm_from_csv_row(row, n_states, n_components, n_features)
            hmms.append(hmm)
            s_type = str(row[COL_SUBJECT_TYPE]).strip().upper()
            labels.append(1 if s_type == ADHD_LABEL.upper() else 0)
            sids.append(str(row[COL_SUBJECT_ID]))
        except Exception as exc:
            print(f"  [WARNING] Skipping row {idx}: {exc}")

    labels = np.array(labels, dtype=int)
    sids   = np.array(sids)
    print(f"  HMMs extracted: {len(hmms)}")
    return hmms, labels, sids


# =============================================================================
# Metric computation (identical to RKHS script)
# =============================================================================

def compute_metrics(y_true, y_pred, y_prob):
    cm             = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sensitivity    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity    = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {
        'accuracy'         : float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'sensitivity'      : float(sensitivity),
        'specificity'      : float(specificity),
        'precision'        : float(precision_score(y_true, y_pred, zero_division=0)),
        'recall'           : float(recall_score(y_true, y_pred, zero_division=0)),
        'f1'               : float(f1_score(y_true, y_pred, zero_division=0)),
        'auc_roc'          : float(roc_auc_score(y_true, y_prob)),
        'mcc'              : float(matthews_corrcoef(y_true, y_pred)),
        'confusion_matrix' : cm,
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
        'n_test'           : len(y_true),
        'n_adhd_test'      : int(y_true.sum()),
        'n_control_test'   : int((1 - y_true).sum()),
    }


def bootstrap_ci(values, n_bootstrap=N_BOOTSTRAP, alpha=CI_ALPHA, seed=0):
    rng    = np.random.default_rng(seed)
    values = np.asarray(values)
    boots  = np.array([
        rng.choice(values, size=len(values), replace=True).mean()
        for _ in range(n_bootstrap)
    ])
    lo = np.percentile(boots, 100 * (1 - alpha) / 2)
    hi = np.percentile(boots, 100 * (1 + alpha) / 2)
    return float(values.mean()), float(lo), float(hi)


# =============================================================================
# Inner grid search
#
# For KNN: grid over (T, rho, k)
# For SVM: grid over (T, rho, C) — K_norm used directly as kernel
#
# Similarity matrix computed once per (T, rho) pair and reused across
# all k / C combinations — same caching strategy as RKHS script.
# =============================================================================

def inner_grid_search_knn(train_hmms, train_labels,
                          T_grid, rho_grid, k_grid,
                          n_inner_splits, inner_seed, fold_id):
    skf     = StratifiedKFold(n_splits=n_inner_splits, shuffle=True,
                              random_state=inner_seed)
    records = []

    for T in T_grid:
        for rho in rho_grid:
            K_inner = compute_similarity_matrix(train_hmms, T, rho, verbose=False)
            D_inner = similarity_to_distance(K_inner)
            for k in k_grid:
                fold_scores = []
                for tr_idx, va_idx in skf.split(train_hmms, train_labels):
                    tr_D = D_inner[np.ix_(tr_idx, tr_idx)]
                    va_D = D_inner[np.ix_(va_idx, tr_idx)]
                    knn  = KNeighborsClassifier(
                        n_neighbors=k, metric='precomputed', weights='distance'
                    )
                    knn.fit(tr_D, train_labels[tr_idx])
                    pred = knn.predict(va_D)
                    fold_scores.append(
                        balanced_accuracy_score(train_labels[va_idx], pred)
                    )
                records.append({
                    'T': T, 'rho': rho, 'k': k,
                    'mean_bacc': float(np.mean(fold_scores)),
                    'std_bacc' : float(np.std(fold_scores)),
                })

    results_df = pd.DataFrame(records).sort_values(
        'mean_bacc', ascending=False).reset_index(drop=True)
    best = results_df.iloc[0]
    print(f"       [Inner-PPK-KNN {fold_id}] Best: "
          f"T={int(best['T'])}  rho={best['rho']:.4g}  k={int(best['k'])}  "
          f"val_bacc={best['mean_bacc']:.4f}")
    return (int(best['T']), float(best['rho']), int(best['k']),
            float(best['mean_bacc']), results_df)


def inner_grid_search_svm(train_hmms, train_labels,
                          T_grid, rho_grid, C_grid,
                          n_inner_splits, inner_seed, fold_id):
    """
    Grid search for PPK-SVM: exhaustive search over (T, rho, C).

    K_norm (the normalised PPK similarity matrix) is used directly as the
    SVM kernel after PSD correction — no gamma envelope is needed since
    PPK similarity is already a bounded [0,1] Gram matrix, unlike RKHS's
    unbounded MMD2 distance.
    """
    skf     = StratifiedKFold(n_splits=n_inner_splits, shuffle=True,
                              random_state=inner_seed)
    records = []

    for T in T_grid:
        for rho in rho_grid:
            K_sim = compute_similarity_matrix(train_hmms, T, rho, verbose=False)
            K_reg = kernel_to_svm_kernel(K_sim)
            for C in C_grid:
                fold_scores = []
                for tr_idx, va_idx in skf.split(train_hmms, train_labels):
                    tr_K = K_reg[np.ix_(tr_idx, tr_idx)]
                    va_K = K_reg[np.ix_(va_idx, tr_idx)]
                    svm  = SVC(kernel='precomputed', C=C,
                               probability=True, random_state=inner_seed)
                    svm.fit(tr_K, train_labels[tr_idx])
                    pred = svm.predict(va_K)
                    fold_scores.append(
                        balanced_accuracy_score(train_labels[va_idx], pred)
                    )
                records.append({
                    'T': T, 'rho': rho, 'C': C,
                    'mean_bacc': float(np.mean(fold_scores)),
                    'std_bacc' : float(np.std(fold_scores)),
                })

    results_df = pd.DataFrame(records).sort_values(
        'mean_bacc', ascending=False).reset_index(drop=True)
    best = results_df.iloc[0]
    print(f"       [Inner-PPK-SVM {fold_id}] Best: "
          f"T={int(best['T'])}  rho={best['rho']:.4g}  "
          f"C={best['C']:.4g}  val_bacc={best['mean_bacc']:.4f}")
    return (int(best['T']), float(best['rho']),
            float(best['C']), float(best['mean_bacc']), results_df)


# =============================================================================
# Nested cross-validation — PPK-KNN
# =============================================================================

def nested_cv_knn(hmms, labels, subject_ids,
                  T_grid, rho_grid, k_grid,
                  n_outer_splits, n_outer_repeats, n_inner_splits,
                  outer_cv_base_seed, checkpoint_mgr=None):
    n_total      = n_outer_splits * n_outer_repeats
    fold_records = []
    subj_tracker = defaultdict(
        lambda: {'n_test': 0, 'n_correct': 0, 'predictions': [], 'true': -1}
    )
    global_fold     = 0
    completed_folds = set()

    # ---- Resume from checkpoint if available --------------------------------
    if checkpoint_mgr is not None:
        state = checkpoint_mgr.load()
        if state is not None and state.get('phase') == 'nested_cv':
            fold_records   = state['fold_records']
            subj_tracker   = defaultdict(
                lambda: {'n_test': 0, 'n_correct': 0,
                         'predictions': [], 'true': -1},
                state['subj_tracker'],
            )
            completed_folds = {r['fold_id'] for r in fold_records}
            print(f"  [checkpoint] Resuming: {len(completed_folds)}/{n_total} "
                  f"folds already completed")

    print("\n" + "=" * 70)
    print(f"  NESTED CV — PPK-KNN  [{CONFIG_TAG}]")
    print(f"  {n_outer_splits} folds × {n_outer_repeats} repeats = {n_total} evaluations")
    print(f"  Inner grid: T({len(T_grid)}) × rho({len(rho_grid)}) × k({len(k_grid)}) "
          f"= {len(T_grid)*len(rho_grid)*len(k_grid)} combinations per fold")
    print(f"  Similarity matrices per fold: {len(T_grid)*len(rho_grid)} "
          f"(one per (T,rho) pair)")
    print("=" * 70)

    for repeat in range(n_outer_repeats):
        outer_seed = outer_cv_base_seed + repeat
        skf        = StratifiedKFold(n_splits=n_outer_splits, shuffle=True,
                                     random_state=outer_seed)
        print(f"\n  -- Repeat {repeat+1}/{n_outer_repeats}  (seed={outer_seed}) --")

        for fold, (train_idx, test_idx) in enumerate(skf.split(hmms, labels)):
            global_fold += 1
            fold_id      = f"r{repeat+1}f{fold+1}"

            if fold_id in completed_folds:
                print(f"\n     Fold {fold+1}/{n_outer_splits}  "
                      f"[global {global_fold}/{n_total}]  id={fold_id}  "
                      f"[SKIP - already completed]")
                continue

            tr_hmms      = [hmms[i] for i in train_idx]
            te_hmms      = [hmms[i] for i in test_idx]
            tr_labels    = labels[train_idx]
            te_labels    = labels[test_idx]

            print(f"\n     Fold {fold+1}/{n_outer_splits}  "
                  f"[global {global_fold}/{n_total}]  id={fold_id}")
            print(f"       Train: {len(train_idx)}  "
                  f"(Control={(tr_labels==0).sum()}  ADHD={(tr_labels==1).sum()})")
            print(f"       Test : {len(test_idx)}  "
                  f"(Control={(te_labels==0).sum()}  ADHD={(te_labels==1).sum()})")

            best_T, best_rho, best_k, inner_score, _ = inner_grid_search_knn(
                tr_hmms, tr_labels, T_grid, rho_grid, k_grid,
                n_inner_splits, outer_seed, fold_id,
            )

            all_hmms = tr_hmms + te_hmms
            K_full   = compute_similarity_matrix(all_hmms, best_T, best_rho,
                                                 verbose=False)
            D_full   = similarity_to_distance(K_full)
            n_tr     = len(tr_hmms)
            tr_D     = D_full[:n_tr, :n_tr]
            te_D     = D_full[n_tr:, :n_tr]

            knn = KNeighborsClassifier(
                n_neighbors=best_k, metric='precomputed', weights='distance'
            )
            knn.fit(tr_D, tr_labels)
            te_pred = knn.predict(te_D)
            te_prob = knn.predict_proba(te_D)[:, 1]

            metrics = compute_metrics(te_labels, te_pred, te_prob)
            metrics.update({
                'config_tag'    : CONFIG_TAG,
                'classifier'    : 'PPK-KNN',
                'fold_id'       : fold_id,
                'repeat'        : repeat,
                'fold'          : fold,
                'outer_seed'    : outer_seed,
                'best_T'        : best_T,
                'best_rho'      : best_rho,
                'best_k'        : best_k,
                'inner_best_val': inner_score,
                'n_train'       : len(train_idx),
                'n_train_ctrl'  : int((tr_labels == 0).sum()),
                'n_train_adhd'  : int((tr_labels == 1).sum()),
            })
            fold_records.append(metrics)

            print(f"       [PPK-KNN] Acc={metrics['accuracy']:.4f}  "
                  f"BalAcc={metrics['balanced_accuracy']:.4f}  "
                  f"AUC={metrics['auc_roc']:.4f}  "
                  f"Sens={metrics['sensitivity']:.4f}  "
                  f"Spec={metrics['specificity']:.4f}  "
                  f"F1={metrics['f1']:.4f}  MCC={metrics['mcc']:.4f}")

            for i, gi in enumerate(test_idx):
                sid = subject_ids[gi]
                subj_tracker[sid]['n_test']    += 1
                subj_tracker[sid]['n_correct'] += int(te_pred[i] == te_labels[i])
                subj_tracker[sid]['predictions'].append(int(te_pred[i]))
                subj_tracker[sid]['true']       = int(te_labels[i])

            # ---- Checkpoint after every completed fold --------------------------
            if checkpoint_mgr is not None:
                checkpoint_mgr.save({
                    'phase'       : 'nested_cv',
                    'fold_records': fold_records,
                    'subj_tracker': dict(subj_tracker),
                })
                print(f"       [checkpoint] Saved progress "
                      f"({len(fold_records)}/{n_total} folds)")

    return fold_records, dict(subj_tracker)


# =============================================================================
# Nested cross-validation — PPK-SVM
# =============================================================================

def nested_cv_svm(hmms, labels, subject_ids,
                  T_grid, rho_grid, C_grid,
                  n_outer_splits, n_outer_repeats, n_inner_splits,
                  outer_cv_base_seed, checkpoint_mgr=None):
    n_total      = n_outer_splits * n_outer_repeats
    fold_records = []
    subj_tracker = defaultdict(
        lambda: {'n_test': 0, 'n_correct': 0, 'predictions': [], 'true': -1}
    )
    global_fold     = 0
    completed_folds = set()

    # ---- Resume from checkpoint if available --------------------------------
    if checkpoint_mgr is not None:
        state = checkpoint_mgr.load()
        if state is not None and state.get('phase') == 'nested_cv':
            fold_records   = state['fold_records']
            subj_tracker   = defaultdict(
                lambda: {'n_test': 0, 'n_correct': 0,
                         'predictions': [], 'true': -1},
                state['subj_tracker'],
            )
            completed_folds = {r['fold_id'] for r in fold_records}
            print(f"  [checkpoint] Resuming: {len(completed_folds)}/{n_total} "
                  f"folds already completed")

    print("\n" + "=" * 70)
    print(f"  NESTED CV — PPK-SVM  [{CONFIG_TAG}]")
    print(f"  {n_outer_splits} folds × {n_outer_repeats} repeats = {n_total} evaluations")
    print(f"  Inner grid: T({len(T_grid)}) × rho({len(rho_grid)}) × "
          f"C({len(C_grid)}) = "
          f"{len(T_grid)*len(rho_grid)*len(C_grid)} combos per fold")
    print(f"  Similarity matrices per fold: {len(T_grid)*len(rho_grid)} "
          f"(one per (T,rho) pair)")
    print(f"  Note: K_norm used directly as SVM kernel (no gamma envelope)")
    print("=" * 70)

    for repeat in range(n_outer_repeats):
        outer_seed = outer_cv_base_seed + repeat
        skf        = StratifiedKFold(n_splits=n_outer_splits, shuffle=True,
                                     random_state=outer_seed)
        print(f"\n  -- Repeat {repeat+1}/{n_outer_repeats}  (seed={outer_seed}) --")

        for fold, (train_idx, test_idx) in enumerate(skf.split(hmms, labels)):
            global_fold += 1
            fold_id      = f"r{repeat+1}f{fold+1}"

            if fold_id in completed_folds:
                print(f"\n     Fold {fold+1}/{n_outer_splits}  "
                      f"[global {global_fold}/{n_total}]  id={fold_id}  "
                      f"[SKIP - already completed]")
                continue

            tr_hmms      = [hmms[i] for i in train_idx]
            te_hmms      = [hmms[i] for i in test_idx]
            tr_labels    = labels[train_idx]
            te_labels    = labels[test_idx]

            print(f"\n     Fold {fold+1}/{n_outer_splits}  "
                  f"[global {global_fold}/{n_total}]  id={fold_id}")
            print(f"       Train: {len(train_idx)}  "
                  f"(Control={(tr_labels==0).sum()}  ADHD={(tr_labels==1).sum()})")
            print(f"       Test : {len(test_idx)}  "
                  f"(Control={(te_labels==0).sum()}  ADHD={(te_labels==1).sum()})")

            best_T, best_rho, best_C, inner_score, _ = \
                inner_grid_search_svm(
                    tr_hmms, tr_labels, T_grid, rho_grid, C_grid,
                    n_inner_splits, outer_seed, fold_id,
                )

            all_hmms = tr_hmms + te_hmms
            K_full   = compute_similarity_matrix(all_hmms, best_T, best_rho,
                                                 verbose=False)
            n_tr     = len(tr_hmms)

            K_tr = K_full[:n_tr, :n_tr]
            K_te = K_full[n_tr:, :n_tr]

            K_tr_reg = kernel_to_svm_kernel(K_tr)
            check_kernel_psd(K_tr_reg)

            svm = SVC(kernel='precomputed', C=best_C,
                     probability=True, random_state=outer_seed)
            svm.fit(K_tr_reg, tr_labels)
            te_pred = svm.predict(K_te)
            te_prob = svm.predict_proba(K_te)[:, 1]

            metrics = compute_metrics(te_labels, te_pred, te_prob)
            metrics.update({
                'config_tag'    : CONFIG_TAG,
                'classifier'    : 'PPK-SVM',
                'fold_id'       : fold_id,
                'repeat'        : repeat,
                'fold'          : fold,
                'outer_seed'    : outer_seed,
                'best_T'        : best_T,
                'best_rho'      : best_rho,
                'best_C'        : best_C,
                'inner_best_val': inner_score,
                'n_train'       : len(train_idx),
                'n_train_ctrl'  : int((tr_labels == 0).sum()),
                'n_train_adhd'  : int((tr_labels == 1).sum()),
            })
            fold_records.append(metrics)

            print(f"       [PPK-SVM] Acc={metrics['accuracy']:.4f}  "
                  f"BalAcc={metrics['balanced_accuracy']:.4f}  "
                  f"AUC={metrics['auc_roc']:.4f}  "
                  f"Sens={metrics['sensitivity']:.4f}  "
                  f"Spec={metrics['specificity']:.4f}  "
                  f"F1={metrics['f1']:.4f}  MCC={metrics['mcc']:.4f}")

            for i, gi in enumerate(test_idx):
                sid = subject_ids[gi]
                subj_tracker[sid]['n_test']    += 1
                subj_tracker[sid]['n_correct'] += int(te_pred[i] == te_labels[i])
                subj_tracker[sid]['predictions'].append(int(te_pred[i]))
                subj_tracker[sid]['true']       = int(te_labels[i])

            # ---- Checkpoint after every completed fold --------------------------
            if checkpoint_mgr is not None:
                checkpoint_mgr.save({
                    'phase'       : 'nested_cv',
                    'fold_records': fold_records,
                    'subj_tracker': dict(subj_tracker),
                })
                print(f"       [checkpoint] Saved progress "
                      f"({len(fold_records)}/{n_total} folds)")

    return fold_records, dict(subj_tracker)


# =============================================================================
# Permutation test
# =============================================================================

def permutation_test(hmms, labels, classifier,
                     best_T, best_rho, best_params,
                     n_outer_splits, n_outer_repeats, outer_cv_base_seed,
                     n_permutations, perm_seed, observed_score,
                     checkpoint_mgr=None):
    """
    classifier   : 'PPK-KNN' or 'PPK-SVM'
    best_params  : int k (KNN) or dict {'C'} (SVM)
    checkpoint_mgr : CheckpointManager — saves progress every
                     PERM_CKPT_EVERY permutations, including RNG state
                     for exact bitwise resumability.
    """
    rng         = np.random.default_rng(perm_seed)
    perm_scores = np.zeros(n_permutations, dtype=float)
    start_p     = 0

    # ---- Resume from checkpoint if available --------------------------------
    if checkpoint_mgr is not None:
        state = checkpoint_mgr.load()
        if state is not None and state.get('phase') == 'permutation_test':
            perm_scores = state['perm_scores']
            start_p     = state['next_p']
            rng         = state['rng_state']
            print(f"  [checkpoint] Resuming permutation test from "
                  f"{start_p}/{n_permutations}")

    print(f"\n  Permutation test [{classifier}]: n={n_permutations}  "
          f"T={best_T}  rho={best_rho:.4g}")
    if classifier == 'PPK-KNN':
        print(f"  Fixed params: k={best_params}")
    else:
        print(f"  Fixed params: C={best_params['C']:.4g}")
    print(f"  Observed balanced accuracy: {observed_score:.4f}")
    print("  Pre-computing full similarity matrix...")

    K_full = compute_similarity_matrix(hmms, best_T, best_rho, verbose=True)
    if classifier == 'PPK-KNN':
        D_full = similarity_to_distance(K_full)
    else:
        K_reg = kernel_to_svm_kernel(K_full)

    for p in range(start_p, n_permutations):
        perm_labels = rng.permutation(labels)
        fold_baccs  = []

        for repeat in range(n_outer_repeats):
            skf = StratifiedKFold(
                n_splits=n_outer_splits, shuffle=True,
                random_state=outer_cv_base_seed + repeat,
            )
            for train_idx, test_idx in skf.split(hmms, perm_labels):
                if classifier == 'PPK-KNN':
                    tr_D = D_full[np.ix_(train_idx, train_idx)]
                    te_D = D_full[np.ix_(test_idx,  train_idx)]
                    clf  = KNeighborsClassifier(
                        n_neighbors=best_params,
                        metric='precomputed', weights='distance',
                    )
                    clf.fit(tr_D, perm_labels[train_idx])
                    pred = clf.predict(te_D)
                else:
                    tr_K = K_reg[np.ix_(train_idx, train_idx)]
                    te_K = K_full[np.ix_(test_idx, train_idx)]
                    clf  = SVC(kernel='precomputed', C=best_params['C'],
                              probability=False,
                              random_state=outer_cv_base_seed)
                    clf.fit(tr_K, perm_labels[train_idx])
                    pred = clf.predict(te_K)

                fold_baccs.append(
                    balanced_accuracy_score(perm_labels[test_idx], pred)
                )

        perm_scores[p] = float(np.mean(fold_baccs))

        # ---- Checkpoint every PERM_CKPT_EVERY permutations -------------------
        if checkpoint_mgr is not None and (p + 1) % PERM_CKPT_EVERY == 0:
            checkpoint_mgr.save({
                'phase'      : 'permutation_test',
                'perm_scores': perm_scores,
                'next_p'     : p + 1,
                'rng_state'  : rng,
            })
            print(f"    [checkpoint] Saved progress ({p+1}/{n_permutations})")

        if (p + 1) % 200 == 0:
            running_p = (perm_scores[:p+1] >= observed_score).mean()
            print(f"    {p+1}/{n_permutations}  running p={running_p:.4f}")

    p_value = float((perm_scores >= observed_score).mean())
    sig     = 'significant' if p_value < 0.05 else 'not significant'
    print(f"  p-value = {p_value:.4f}  ({sig} at alpha=0.05)")
    return perm_scores, p_value


# =============================================================================
# Aggregate results
# =============================================================================

METRIC_KEYS = [
    'accuracy', 'balanced_accuracy', 'sensitivity', 'specificity',
    'precision', 'recall', 'f1', 'auc_roc', 'mcc',
]


def aggregate_results(fold_records):
    summary_rows = []
    for mk in METRIC_KEYS:
        vals = np.array([r[mk] for r in fold_records])
        mean, lo, hi = bootstrap_ci(vals)
        summary_rows.append({
            'config_tag': CONFIG_TAG,
            'classifier': fold_records[0].get('classifier', ''),
            'metric'    : mk,
            'mean'      : round(mean, 4),
            'std'       : round(float(vals.std()), 4),
            'ci_lower'  : round(lo, 4),
            'ci_upper'  : round(hi, 4),
            'min'       : round(float(vals.min()), 4),
            'max'       : round(float(vals.max()), 4),
            'n_folds'   : len(vals),
        })

    pooled_cm = sum(r['confusion_matrix'] for r in fold_records)

    fold_df = pd.DataFrame([{
        'config_tag'     : r['config_tag'],
        'classifier'     : r.get('classifier', ''),
        'fold_id'        : r['fold_id'],
        'repeat'         : r['repeat'],
        'fold'           : r['fold'],
        'outer_seed'     : r['outer_seed'],
        'best_T'         : r['best_T'],
        'best_rho'       : r['best_rho'],
        'best_k'         : r.get('best_k', np.nan),
        'best_C'         : r.get('best_C', np.nan),
        'inner_best_val' : r['inner_best_val'],
        'n_test'         : r['n_test'],
        'n_test_control' : r['n_control_test'],
        'n_test_adhd'    : r['n_adhd_test'],
        'n_train'        : r['n_train'],
        'n_train_control': r['n_train_ctrl'],
        'n_train_adhd'   : r['n_train_adhd'],
        'tp': r['tp'], 'tn': r['tn'], 'fp': r['fp'], 'fn': r['fn'],
        **{mk: r[mk] for mk in METRIC_KEYS},
    } for r in fold_records])

    return {'summary': pd.DataFrame(summary_rows), 'fold_df': fold_df,
            'pooled_cm': pooled_cm}


def build_subject_df(subject_tracker, classifier):
    rows = []
    for sid, info in subject_tracker.items():
        preds = info['predictions']
        rows.append({
            'config_tag'          : CONFIG_TAG,
            'classifier'          : classifier,
            'subject_id'          : sid,
            'true_label'          : info['true'],
            'true_class'          : 'ADHD' if info['true'] == 1 else 'Control',
            'n_folds_tested'      : info['n_test'],
            'n_correct'           : info['n_correct'],
            'accuracy_rate'       : round(info['n_correct'] / info['n_test'], 4)
                                    if info['n_test'] > 0 else 0.0,
            'pred_adhd_rate'      : round(sum(preds) / len(preds), 4)
                                    if preds else 0.0,
            'consistently_correct': info['n_correct'] == info['n_test'],
            'never_correct'       : info['n_correct'] == 0,
        })
    return pd.DataFrame(rows).sort_values(
        ['true_class', 'accuracy_rate'], ascending=[True, False]
    ).reset_index(drop=True)


# =============================================================================
# Printing
# =============================================================================

def print_summary(summary_df, pooled_cm, p_value, observed_score,
                  classifier, n_outer_splits, n_outer_repeats):
    n_folds = n_outer_splits * n_outer_repeats
    print("\n" + "=" * 70)
    print(f"  NESTED CV RESULTS [{CONFIG_TAG}] [{classifier}]  "
          f"({n_outer_splits}x{n_outer_repeats} = {n_folds} folds)")
    print(f"  Bootstrap CI: {int(CI_ALPHA*100)}%  (n={N_BOOTSTRAP})")
    print("=" * 70)
    print(f"\n  {'Metric':<22}  {'Mean':>7}  {'Std':>7}  "
          f"{'CI Lower':>9}  {'CI Upper':>9}  {'Min':>7}  {'Max':>7}")
    print("  " + "-" * 68)
    for _, row in summary_df.iterrows():
        print(f"  {row['metric']:<22}  {row['mean']:>7.4f}  {row['std']:>7.4f}  "
              f"{row['ci_lower']:>9.4f}  {row['ci_upper']:>9.4f}  "
              f"{row['min']:>7.4f}  {row['max']:>7.4f}")
    tn, fp = pooled_cm[0,0], pooled_cm[0,1]
    fn, tp = pooled_cm[1,0], pooled_cm[1,1]
    print(f"\n  Pooled confusion matrix (sum over {n_folds} folds, "
          f"total={pooled_cm.sum()} predictions):")
    print(f"                   Pred Control  Pred ADHD")
    print(f"  True Control     {tn:<12}  {fp:<10}")
    print(f"  True ADHD        {fn:<12}  {tp:<10}")
    print(f"\n  Permutation test (n={N_PERMUTATIONS}):")
    print(f"    Observed balanced accuracy : {observed_score:.4f}")
    p_str = '<0.001' if p_value < 0.001 else f'{p_value:.4f}'
    sig   = '*significant*' if p_value < 0.05 else 'not significant'
    print(f"    p-value                    : {p_str}  ({sig} at alpha=0.05)")


# =============================================================================
# Plots
# =============================================================================

def save_plots(agg, perm_scores, p_value, observed_score,
               subject_df, output_dir, classifier):
    os.makedirs(output_dir, exist_ok=True)
    summary_df = agg['summary']
    fold_df    = agg['fold_df']
    pooled_cm  = agg['pooled_cm']
    tag        = f"[{CONFIG_TAG}][{classifier}]"

    # Figure 1 — metric summary with CI
    fig, ax = plt.subplots(figsize=(12, 5))
    x     = np.arange(len(METRIC_KEYS))
    sm    = summary_df.set_index('metric')
    means = sm.loc[METRIC_KEYS, 'mean'].values
    los   = sm.loc[METRIC_KEYS, 'ci_lower'].values
    his   = sm.loc[METRIC_KEYS, 'ci_upper'].values
    ax.bar(x, means, color='#4C72B0', alpha=0.8, zorder=3)
    ax.errorbar(x, means, yerr=np.array([means-los, his-means]),
                fmt='none', color='black', capsize=5, linewidth=1.5, zorder=4)
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace('_', '\n') for m in METRIC_KEYS], fontsize=9)
    ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
    ax.set_title(f"{tag} Metric summary — {int(CI_ALPHA*100)}% bootstrap CI",
                fontsize=11)
    ax.axhline(0.5, color='grey', linewidth=0.8, linestyle='--')
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    fig.savefig(f"{output_dir}/metric_summary.png", dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Figure 2 — per-fold balanced accuracy
    fig, ax = plt.subplots(figsize=(14, 5))
    for rep in fold_df['repeat'].unique():
        sub = fold_df[fold_df['repeat'] == rep].sort_values('fold')
        xr  = sub['fold'].values + rep * (N_OUTER_SPLITS + 1)
        ax.plot(xr, sub['balanced_accuracy'].values, 'o-',
                label=f"Repeat {rep+1}", alpha=0.8, linewidth=1.5)
    ax.axhline(fold_df['balanced_accuracy'].mean(), color='black',
              linewidth=1.0, linestyle='--',
              label=f"Mean={fold_df['balanced_accuracy'].mean():.4f}")
    ax.set_ylabel("Balanced Accuracy"); ax.set_xlabel("Fold (by repeat)")
    ax.set_title(f"{tag} Per-fold balanced accuracy", fontsize=11)
    ax.legend(fontsize=8, framealpha=0.4); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(f"{output_dir}/per_fold_bacc.png", dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Figure 3 — pooled confusion matrix
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(pooled_cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Control', 'ADHD'],
                yticklabels=['Control', 'ADHD'], ax=ax)
    ax.set_title(f"{tag}\nPooled confusion matrix", fontsize=10)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.tight_layout()
    fig.savefig(f"{output_dir}/pooled_cm.png", dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Figure 4 — permutation test
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(perm_scores, bins=40, color='#DD8452', alpha=0.75,
            edgecolor='white', label='Permuted scores')
    p_str = '<0.001' if p_value < 0.001 else f'{p_value:.3f}'
    ax.axvline(observed_score, color='#4C72B0', linewidth=2.5, linestyle='--',
              label=f'Observed={observed_score:.4f}  p={p_str}')
    ax.set_xlabel('Mean balanced accuracy'); ax.set_ylabel('Count')
    ax.set_title(f"{tag} Permutation test  (n={N_PERMUTATIONS})", fontsize=11)
    ax.legend(fontsize=9, framealpha=0.5); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(f"{output_dir}/permutation_test.png", dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Figure 5 — subject stability
    fig, ax = plt.subplots(figsize=(14, 5))
    colors = ['#1D9E75' if c == 'Control' else '#D85A30'
              for c in subject_df['true_class']]
    ax.bar(range(len(subject_df)), subject_df['accuracy_rate'],
           color=colors, alpha=0.8, zorder=3)
    ax.axhline(0.5, color='grey', linewidth=0.8, linestyle='--')
    ax.set_xticks(range(len(subject_df)))
    ax.set_xticklabels(subject_df['subject_id'], rotation=90, fontsize=6)
    ax.set_ylabel("Correct classification rate")
    ax.set_title(f"{tag} Per-subject stability", fontsize=11)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color='#1D9E75', label='Control'),
                       Patch(color='#D85A30', label='ADHD')], fontsize=9)
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    fig.savefig(f"{output_dir}/subject_stability.png", dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Figure 6 — hyperparameter distribution
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].hist(fold_df['best_T'], bins=len(T_GRID), color='#4C72B0',
                 alpha=0.8, edgecolor='white')
    axes[0].set_xlabel('T'); axes[0].set_ylabel('Count')
    axes[0].set_title(f"{tag} Selected T"); axes[0].grid(True, alpha=0.3)

    axes[1].hist(fold_df['best_rho'], bins=len(RHO_GRID), color='#DD8452',
                 alpha=0.8, edgecolor='white')
    axes[1].set_xlabel('rho (ρ)'); axes[1].set_ylabel('Count')
    axes[1].set_title(f"{tag} Selected rho"); axes[1].grid(True, alpha=0.3)

    if classifier == 'PPK-KNN':
        k_counts = fold_df['best_k'].value_counts().sort_index()
        axes[2].bar(k_counts.index.astype(str), k_counts.values,
                   color='#8172B2', alpha=0.8)
        axes[2].set_xlabel('k'); axes[2].set_ylabel('Count')
        axes[2].set_title(f"{tag} Selected k")
        axes[2].grid(True, axis='y', alpha=0.3)
    else:
        C_counts = fold_df['best_C'].value_counts().sort_index()
        axes[2].bar(C_counts.index.astype(str), C_counts.values,
                   color='#55A868', alpha=0.8)
        axes[2].set_xlabel('C'); axes[2].set_ylabel('Count')
        axes[2].set_title(f"{tag} Selected C")
        axes[2].grid(True, axis='y', alpha=0.3)

    plt.tight_layout()
    fig.savefig(f"{output_dir}/hyperparameter_dist.png",
               dpi=150, bbox_inches='tight')
    plt.close(fig)

    print(f"  Figures saved to: {output_dir}")


# =============================================================================
# Run one classifier end-to-end
# =============================================================================

def run_classifier(hmms, labels, subject_ids, classifier, force_rerun=False):
    """
    Full pipeline for one classifier ('PPK-KNN' or 'PPK-SVM'):
        1. Nested CV with inner grid search
        2. Aggregate metrics + bootstrap CI
        3. Permutation test
        4. Save outputs

    If a COMPLETE result already exists on disk (fold_results.csv with
    exactly n_outer_splits * n_outer_repeats rows, plus permutation_scores.csv
    with n_permutations rows), this function skips recomputation entirely
    and loads the existing results instead — unless force_rerun=True.
    """
    assert classifier in ('PPK-KNN', 'PPK-SVM')
    out_dir = os.path.join(OUTPUT_BASE, CONFIG_TAG, classifier)
    os.makedirs(out_dir, exist_ok=True)
    checkpoint_mgr = CheckpointManager(out_dir, classifier)

    # ---- Skip entirely if a complete result already exists -------------------
    n_expected_folds = N_OUTER_SPLITS * N_OUTER_REPEATS
    fold_csv = os.path.join(out_dir, "fold_results.csv")
    perm_csv = os.path.join(out_dir, "permutation_scores.csv")
    perm_txt = os.path.join(out_dir, "permutation_result.txt")

    if not force_rerun and all(os.path.exists(p) for p in
                               (fold_csv, perm_csv, perm_txt)):
        try:
            existing_fold_df = pd.read_csv(fold_csv)
            existing_perm_df = pd.read_csv(perm_csv)
            if (len(existing_fold_df) == n_expected_folds
                    and len(existing_perm_df) == N_PERMUTATIONS):
                print(f"\n  [SKIP] Complete result already found for "
                      f"[{CONFIG_TAG}] [{classifier}] at {out_dir}")
                print(f"         fold_results.csv: {len(existing_fold_df)}/"
                      f"{n_expected_folds} folds")
                print(f"         permutation_scores.csv: "
                      f"{len(existing_perm_df)}/{N_PERMUTATIONS} permutations")
                print(f"         Pass force_rerun=True to recompute anyway.")

                existing_summary_df = pd.read_csv(
                    os.path.join(out_dir, "metric_summary.csv"))
                observed_ba = float(
                    existing_fold_df['balanced_accuracy'].mean())
                with open(perm_txt) as f:
                    txt = f.read()
                p_match = re.search(r'p-value\s*:\s*([0-9.]+)', txt)
                p_value = float(p_match.group(1)) if p_match else float('nan')

                pooled_cm = np.array([
                    [existing_fold_df['tn'].sum(), existing_fold_df['fp'].sum()],
                    [existing_fold_df['fn'].sum(), existing_fold_df['tp'].sum()],
                ])

                print(f"\n  Loaded existing result: "
                      f"balanced_accuracy={observed_ba:.4f}  p={p_value:.4f}")

                return {
                    'fold_records': None,
                    'subject_df'  : pd.read_csv(
                        os.path.join(out_dir, "subject_stability.csv")),
                    'agg': {
                        'summary'  : existing_summary_df,
                        'fold_df'  : existing_fold_df,
                        'pooled_cm': pooled_cm,
                    },
                    'perm_scores': existing_perm_df['balanced_accuracy'].values,
                    'p_value'    : p_value,
                    'observed_ba': observed_ba,
                }
        except Exception as exc:
            print(f"  [WARNING] Could not validate existing result "
                  f"({exc}) — recomputing from scratch.")

    if classifier == 'PPK-KNN':
        fold_records, subj_tracker = nested_cv_knn(
            hmms, labels, subject_ids,
            T_grid=T_GRID, rho_grid=RHO_GRID, k_grid=K_GRID,
            n_outer_splits=N_OUTER_SPLITS,
            n_outer_repeats=N_OUTER_REPEATS,
            n_inner_splits=N_INNER_SPLITS,
            outer_cv_base_seed=OUTER_CV_BASE_SEED,
            checkpoint_mgr=checkpoint_mgr,
        )
    else:
        fold_records, subj_tracker = nested_cv_svm(
            hmms, labels, subject_ids,
            T_grid=T_GRID, rho_grid=RHO_GRID, C_grid=C_GRID,
            n_outer_splits=N_OUTER_SPLITS,
            n_outer_repeats=N_OUTER_REPEATS,
            n_inner_splits=N_INNER_SPLITS,
            outer_cv_base_seed=OUTER_CV_BASE_SEED,
            checkpoint_mgr=checkpoint_mgr,
        )

    agg         = aggregate_results(fold_records)
    subject_df  = build_subject_df(subj_tracker, classifier)
    observed_ba = float(agg['fold_df']['balanced_accuracy'].mean())

    best_T_perm   = int(agg['fold_df']['best_T'].mode()[0])
    best_rho_perm = float(agg['fold_df']['best_rho'].median())

    if classifier == 'PPK-KNN':
        best_k_perm      = int(agg['fold_df']['best_k'].mode()[0])
        perm_params      = best_k_perm
        perm_params_desc = (f"T={best_T_perm}  rho={best_rho_perm:.4g}  "
                            f"k={best_k_perm}")
    else:
        best_C_perm      = float(agg['fold_df']['best_C'].mode()[0])
        perm_params      = {'C': best_C_perm}
        perm_params_desc = (f"T={best_T_perm}  rho={best_rho_perm:.4g}  "
                            f"C={best_C_perm:.4g}")

    print(f"\n  Permutation test hyperparams: {perm_params_desc}")

    perm_scores, p_value = permutation_test(
        hmms, labels, classifier,
        best_T=best_T_perm, best_rho=best_rho_perm,
        best_params=perm_params,
        n_outer_splits=N_OUTER_SPLITS,
        n_outer_repeats=N_OUTER_REPEATS,
        outer_cv_base_seed=OUTER_CV_BASE_SEED,
        n_permutations=N_PERMUTATIONS,
        perm_seed=PERMUTATION_SEED,
        observed_score=observed_ba,
        checkpoint_mgr=checkpoint_mgr,
    )

    # ---- All phases complete — clear checkpoint ------------------------------
    checkpoint_mgr.clear()
    print(f"  [checkpoint] Run complete — checkpoint cleared")

    print_summary(agg['summary'], agg['pooled_cm'], p_value, observed_ba,
                 classifier, N_OUTER_SPLITS, N_OUTER_REPEATS)

    agg['summary'].to_csv(f"{out_dir}/metric_summary.csv", index=False)
    agg['fold_df'].to_csv(f"{out_dir}/fold_results.csv", index=False)
    subject_df.to_csv(f"{out_dir}/subject_stability.csv", index=False)

    perm_df = pd.DataFrame({
        'config_tag'       : CONFIG_TAG,
        'classifier'       : classifier,
        'permutation_index': np.arange(N_PERMUTATIONS),
        'balanced_accuracy': perm_scores,
    })
    perm_df.to_csv(f"{out_dir}/permutation_scores.csv", index=False)

    with open(f"{out_dir}/permutation_result.txt", 'w') as f:
        f.write(f"Config tag                 : {CONFIG_TAG}\n")
        f.write(f"Classifier                 : {classifier}\n")
        f.write(f"Observed balanced accuracy : {observed_ba:.6f}\n")
        f.write(f"n_permutations             : {N_PERMUTATIONS}\n")
        f.write(f"p-value                    : {p_value:.6f}\n")
        f.write(f"Significant at 0.05        : {p_value < 0.05}\n")
        f.write(f"Perm test params           : {perm_params_desc}\n")
        f.write(f"Note: PPK uses INITIAL pi (not stationary) and the single\n")
        f.write(f"best-weight GMM component per state, per the original\n")
        f.write(f"PPK formulation between HMMs.\n")

    save_plots(agg, perm_scores, p_value, observed_ba,
              subject_df, out_dir, classifier)

    print(f"\n  Outputs saved to: {out_dir}")
    return {
        'fold_records': fold_records, 'subject_df': subject_df,
        'agg': agg, 'perm_scores': perm_scores,
        'p_value': p_value, 'observed_ba': observed_ba,
    }


# =============================================================================
# Entry point
# =============================================================================

if __name__ == "__main__":

    print("=" * 70)
    print(f"  PPK CLASSIFIERS — KNN and SVM  [{CONFIG_TAG}]")
    print(f"  Kernel: Probability Product Kernel (PPK) between HMMs")
    print(f"  Uses INITIAL pi (not stationary), best-weight GMM component "
          f"per state")
    print(f"  Notation: T = transition propagation steps; "
          f"rho (ρ) = PPK exponent")
    print(f"  Note: PPK-SVM uses the normalised similarity matrix K_norm "
          f"directly as the kernel (no gamma envelope needed — K_norm is "
          f"already a bounded [0,1] Gram matrix)")
    print(f"  T grid    : {T_GRID}")
    print(f"  rho grid  : {[f'{v:.3g}' for v in RHO_GRID]}")
    print(f"  C grid    : {C_GRID}")
    print(f"  k grid    : {K_GRID}")
    print("=" * 70)

    hmms, labels, subject_ids = load_dataset(
        csv_path=CSV_PATH, n_states=N_STATES,
        n_components=N_COMPONENTS, n_features=N_FEATURES,
    )

    results_knn = run_classifier(hmms, labels, subject_ids, 'PPK-KNN')
    results_svm = run_classifier(hmms, labels, subject_ids, 'PPK-SVM')

    combined_df = pd.concat([
        results_knn['agg']['summary'],
        results_svm['agg']['summary'],
    ], ignore_index=True)
    combined_path = os.path.join(OUTPUT_BASE, CONFIG_TAG, 'combined_summary.csv')
    combined_df.to_csv(combined_path, index=False)

    print("\n" + "=" * 70)
    print(f"  COMPLETE  [{CONFIG_TAG}]")
    print("=" * 70)
    print(f"  PPK-KNN  balanced accuracy: {results_knn['observed_ba']:.4f}  "
          f"p={results_knn['p_value']:.4f}")
    print(f"  PPK-SVM  balanced accuracy: {results_svm['observed_ba']:.4f}  "
          f"p={results_svm['p_value']:.4f}")
    print(f"  Combined summary -> {combined_path}")